# Voorspel verhuuraantallen 2026 – CityBike

**Machine Learning ADS&AI 2026-27 · Portfolio-opdracht 2**
De Haagse Hogeschool

| Teamgegevens | |
|---|---|
| **Teamnaam op Kaggle** | `[teamnaam invullen]` |
| **Competitie** | [Voorspel verhuuraantallen 2026](https://www.kaggle.com/competitions/voorspel-verhuuraantallen-2026) |
| **Datum** | november 2026 |

| Naam | Studentnummer | Kaggle-gebruikersnaam |
|---|---|---|
| `[Baset Paiman]` | `[24071005]` | `[basetpaimam]` |
| `[Alisha]` | `[nummer]` | `[gebruikersnaam]` |
| `[Shalini]` | `[nummer]` | `[gebruikersnaam]` |

## Inhoudsopgave

1. Inleiding
2. Voorbereiding: packages, instellingen en hulpfuncties

## 1. Inleiding

### 1.1 Context en doel

CityBike verhuurt fietsen op verschillende locaties in een grote stad. Staan
er te weinig fietsen, dan kunnen klanten niet huren. Staan er te veel, dan
staan fietsen en stallingsplekken voor niets stil. Als CityBike vooraf weet
hoeveel fietsen er per uur nodig zijn, kan het bedrijf de fietsen slimmer
verdelen en onderhoud plannen op rustige momenten.

**Onze opdracht:** voorspel voor elk uur van 1 t/m 19 december 2012 hoeveel
fietsen er worden verhuurd (kolom `cnt`). We hebben daarvoor uurdata van
1 januari 2011 t/m 30 november 2012 met informatie over het weer, feestdagen
en het tijdstip.

### 1.2 Aanpak en leeswijzer

We volgen de stappen van het data-scienceproces:

| Hoofdstuk | Wat doen we? |
|---|---|
| 3. EDA | Data leren kennen, fouten opsporen en opschonen, patronen zoeken. |
| 4. Feature engineering | Trend, seizoenen (Fourier), cycli en autocorrelatie onderzoeken en omzetten in features. We testen elke groep features met cross-validatie. |
| 5. Modelleren | Zes machine-learningmodellen, twee tijdreeksmodellen (SARIMAX en Prophet), een hybride model en een stacking-ensemble. Elk model krijgt cross-validatie en hyperparameter tuning. |
| 6. Bevindingen | Resultaten naast elkaar en de Kaggle-scores. |
| 7. Conclusie | Belangrijkste features, advies aan CityBike en modelkeuze. |

Feature engineering en modelleren deden we in de praktijk niet netjes na
elkaar maar in rondjes: feature toevoegen → model testen → terug naar de
features. In paragraaf 4.7 laten we die iteraties zien.

### 1.3 Evaluatiemaat: RMSE

Kaggle beoordeelt de voorspellingen met de **Root Mean Squared Error**:

$$
\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2}
$$

Hierin is $y_i$ het echte aantal verhuurde fietsen in uur $i$ en
$\hat{y}_i$ onze voorspelling. De RMSE is in dezelfde eenheid als de target
(fietsen per uur). Omdat de fouten gekwadrateerd worden, tellen grote fouten
extra zwaar mee. Voor CityBike is dat logisch: een flinke misser in de
ochtendspits (lege stations) is erger dan een paar fietsen verschil midden in
de nacht. **De eis voor deze opdracht is een RMSE onder de 80 op Kaggle.**


## 2. Voorbereiding

### 2.1 Packages en instellingen

Naast de standaardpackages gebruiken we:

* **statsmodels** voor tijdreeksanalyse (ACF/PACF, decompositie, SARIMAX);
* **lightgbm** voor gradient boosting;
* **prophet** als tweede tijdreeksmodel.

Daarnaast natuurlijk pandas, numpy, matplotlib en seaborn, en scikit-learn
voor alle ML-modellen (Pedregosa et al., 2011). statsmodels is beschreven door
Seabold & Perktold (2010).

In [3]:
# Standaardbibliotheek
import logging
import time
import warnings
from pathlib import Path

# Data en rekenen
import numpy as np
import pandas as pd
from scipy import signal

# Visualisatie
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

# Machine learning
import lightgbm as lgb
from joblib import Parallel, delayed
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import (ElasticNet, Lasso, LinearRegression,
                                  PoissonRegressor, Ridge)
from sklearn.metrics import make_scorer
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
                                     TimeSeriesSplit)
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree

# Tijdreeksen
from prophet import Prophet
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.sm_exceptions import (ConvergenceWarning
                                             as SmConvergenceWarning,
                                             InterpolationWarning,
                                             ValueWarning)
from statsmodels.tsa.seasonal import MSTL, STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf, adfuller, kpss, pacf

In [4]:
# Waarschuwingen die we bewust negeren (ze veranderen niets aan de uitkomst)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=SmConvergenceWarning)
warnings.filterwarnings('ignore', category=InterpolationWarning)
warnings.filterwarnings('ignore', category=ValueWarning)
warnings.filterwarnings('ignore', message='.*does not have valid feature')
warnings.filterwarnings('ignore', message='IProgress not found')
warnings.filterwarnings('ignore', message='A worker stopped')
logging.getLogger('cmdstanpy').disabled = True
logging.getLogger('prophet').setLevel(logging.ERROR)

# Vaste instellingen voor het hele notebook
RANDOM_STATE = 42             # vaste seed: resultaten zijn reproduceerbaar
HORIZON_HOURS = 19 * 24       # testperiode: 1 t/m 19 december = 456 uur
N_SPLITS = 5                  # aantal folds in de tijdreeks-CV
N_JOBS = -1                   # gebruik alle processorkernen bij het tunen
OUTAGE_DAYS = ['2012-10-29', '2012-10-30']   # systeem dicht (orkaan Sandy)
SUBMISSION_DIR = Path('submissions')
RESULTS_DIR = Path('resultaten')
SUBMISSION_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)
np.random.seed(RANDOM_STATE)

### 2.2 Hulpfuncties voor figuren en evaluatie

**Figuren.** We geven alle figuren dezelfde rustige stijl: dunne lijnen,
lichte rasterlijnen en steeds dezelfde kleurvolgorde. De kleuren zijn getest
op kleurenblindheid en voor heatmaps gebruiken we een kleur van licht naar
donker (meer = donkerder) en voor correlaties twee kleuren met grijs in het
midden.

**Evaluatie.** De functie `rmse` berekent de Kaggle maat en een aantal fietsen
kan niet negatief zijn dus negatieve voorspellingen zetten we op 0 voordat we
de score berekenen (`clipped_rmse`). De klasse `ResultsTracker` bewaart per
model de scores per fold de gekozen hyperparameters en de voorspellingen.
Zo vergelijken we alle modellen op precies dezelfde manier.

In [6]:
PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100',
           '#e87ba4', '#008300', '#6250d6', '#e34948']
INK = {'primary': '#0b0b0b', 'secondary': '#52514e', 'muted': '#898781',
       'grid': '#e1e0d9', 'axis': '#c3c2b7', 'surface': '#fcfcfb'}
SEQ_CMAP = LinearSegmentedColormap.from_list(
    'blauw', ['#cde2fb', '#86b6ef', '#3987e5', '#1c5cab', '#0d366b'])
DIV_CMAP = LinearSegmentedColormap.from_list(
    'blauw_rood', ['#2a78d6', '#f0efec', '#e34948'])
DAY_NAMES = ['ma', 'di', 'wo', 'do', 'vr', 'za', 'zo']


def set_plot_style():
    """Zet één consistente, rustige stijl voor alle figuren."""
    sns.set_theme(style='whitegrid')
    plt.rcParams.update({
        'figure.figsize': (11, 4.2),
        'figure.dpi': 100,
        'figure.facecolor': INK['surface'],
        'axes.facecolor': INK['surface'],
        'axes.edgecolor': INK['axis'],
        'axes.labelcolor': INK['secondary'],
        'axes.titlecolor': INK['primary'],
        'axes.titlesize': 12,
        'axes.titleweight': 'bold',
        'axes.titlelocation': 'left',
        'axes.labelsize': 10,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.grid': True,
        'axes.prop_cycle': plt.cycler(color=PALETTE),
        'grid.color': INK['grid'],
        'grid.linestyle': '-',
        'grid.linewidth': 0.6,
        'xtick.color': INK['muted'],
        'ytick.color': INK['muted'],
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.frameon': False,
        'legend.fontsize': 9,
        'lines.linewidth': 1.6,
        'lines.solid_capstyle': 'round',
    })


DUTCH_MONTHS = ['jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug',
                'sep', 'okt', 'nov', 'dec']


def finish_axis(ax, title, xlabel='', ylabel=''):
    """Zet titel en aslabels op een vaste manier."""
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)


def dutch_dates(ax, style='maand_jaar'):
    """Nederlandse datumlabels op de x-as (maand_jaar, maand, dag, weekdag)."""
    def label(value, _position):
        date = mdates.num2date(value)
        month = DUTCH_MONTHS[date.month - 1]
        formats = {'maand_jaar': f'{month} {date.year}',
                   'maand': month,
                   'dag': f'{date.day} {month}',
                   'weekdag': f'{DAY_NAMES[date.weekday()]} {date.day} '
                              f'{month}'}
        return formats[style]
    ax.xaxis.set_major_formatter(plt.FuncFormatter(label))


set_plot_style()